In [ ]:
import numpy as np


def structure(value):
    def decorator(func):
        func.structure = value
        return func

    return decorator


def changes(l, rng=None):
    rng = rng or np.random.default_rng()
    changes = []
    for _ in range(l):
        if rng.random() < 0.1:
            n = int(rng.integers(50, 1000))
            P = int(rng.integers(0, 3))
        else:
            n = int(rng.integers(10, 60))
            P = int(rng.integers(0, 5))
        positives = np.arange(P)
        changes.append((positives, n))
    return changes


def predictions(p, n, mu=None, reps=1, rng=None):
    rng = rng or np.random.default_rng()
    if mu is None:
        mu = np.linspace(0.0, 4.0, 40)

    p = np.asarray(p)
    if p.dtype == bool:
        pos = p
    else:
        pos = np.isin(np.arange(n), p.astype(int))

    mus = np.repeat(mu, reps)
    base = rng.standard_normal(n)
    scores = np.tile(base, (mus.size, 1))
    scores[:, pos] += mus[:, None]
    return scores


def topk(order, k):
    pred = np.zeros(order.shape, bool)
    if k > 0:
        pred[np.arange(order.shape[0])[:, None], order[:, :k]] = True
    return pred


def threshold(metric, S, y):
    order = np.argsort(-S, axis=1)
    n_mu, n = S.shape
    vals = np.column_stack([metric(topk(order, k), y) for k in range(n + 1)])
    return vals.argmax(axis=1)


def score(metric, S, kstar, y):
    order = np.argsort(-S, axis=1)
    pred = np.zeros(S.shape, bool)
    for r, k in enumerate(kstar):
        if k > 0:
            pred[r, order[r, :k]] = True
    return metric(pred, y)


@structure("label")
def f1(pred, y):
    TP = (pred & y).sum(axis=1)
    FP = (pred & ~y).sum(axis=1)
    FN = (~pred & y).sum(axis=1)
    den = 2 * TP + FP + FN
    return np.where(den > 0, 2 * TP / np.maximum(den, 1), 1.0)


@structure("label")
def mcc(pred, y):
    TP = (pred & y).sum(axis=1).astype(float)
    FP = (pred & ~y).sum(axis=1).astype(float)
    FN = (~pred & y).sum(axis=1).astype(float)
    TN = (~pred & ~y).sum(axis=1).astype(float)
    num = TP * TN - FP * FN
    den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    return np.where(den > 0, num / np.maximum(den, 1e-12), 0.0)


def theta_star(
    C, mu, metric=f1, M=300, alpha=(1, 1), beta=(1, 1), delta=(1, 1), rng=None
):
    rng = rng or np.random.default_rng()
    S = [predictions(p, n, mu, rng=rng) for p, n in C]
    Y = [np.isin(np.arange(n), list(p)) for p, n in C]

    thresholds = (
        [None] * len(C)
        if metric.structure != "label"
        else [threshold(metric, s, y) for s, y in zip(S, Y)]
    )
    level = np.mean(
        [
            (
                metric(S[i], Y[i])
                if metric.structure != "label"
                else score(metric, S[i], thresholds[i], Y[i])
            )
            for i in range(len(C))
        ],
        axis=0,
    )

    agg = np.empty((M, mu.size))
    for m in range(M):
        a = rng.beta(*alpha)
        b = rng.beta(*beta)
        d = rng.beta(*delta)
        per_change = np.empty((len(C), mu.size))
        for i, (s, y) in enumerate(zip(S, Y)):
            P, n = int(y.sum()), y.size
            if P > 0:
                ym = y.copy()
                ym[y] = rng.random(P) > a
            elif rng.random() < b:
                ym = rng.random(n) < d
            else:
                ym = np.zeros(n, bool)
            per_change[i] = (
                metric(S[i], ym)
                if metric.structure != "label"
                else score(metric, S[i], thresholds[i], ym)
            )
        agg[m] = per_change.mean(axis=0)
    lo, hi = np.quantile(agg, [0.025, 0.975], axis=0)
    kappa = hi - lo
    return level, kappa, kappa.max()

In [ ]:
import plotly.graph_objects as go

rng = np.random.default_rng()
C = changes(300, rng)
n_max = max(n for p, n in C if len(p) > 0)
mu = np.linspace(0.0, np.sqrt(2 * np.log(n_max)) + 2.5, 50)


level, kappa, theta = theta_star(
    C, mu, metric=f1, alpha=(2, 18), beta=(7, 3), delta=(4, 4), M=500, rng=rng
)

o = np.argsort(level)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=level[o],
        y=kappa[o],
        mode="lines+markers",
        name="κ(level)",
        line=dict(color="steelblue"),
        marker=dict(size=6),
    )
)
fig.add_hline(
    y=theta,
    line_dash="dash",
    line_color="crimson",
    annotation_text=f"θ* = max κ = {theta:.3f}",
    annotation_position="top left",
)
fig.update_layout(
    xaxis_title="Metric",
    yaxis_title="κ",
    template="simple_white",
)
fig.show()

: 